In [1]:
from __future__ import annotations

from collections import defaultdict
from datetime import date, datetime, timedelta
from pathlib import Path
import gc
import shutil
from typing import Any

import pythoncom
import win32com.client as win32

SAMPLES_DIR = Path("samples")
OUTPUT_DIR = Path("outputs")
MASTER_FILE = SAMPLES_DIR / "Expense Report Relating to deals 01-01-26 to 6-30-26_V8 - COMBINED - Try SAP.xlsx"
HR_FILE = SAMPLES_DIR / "Headcount CC 0426.xlsx"
SOURCE_SHEET_NAME = "To be Copy Pasted from HR"
MASTER_SHEET_PREFIX = "HR HC Combined"
NETWORK_ID_HEADER = "Network ID"
DUPLICATE_HEADERS = {"DUPLICATE", "DUPLICATES", "DUPLICATE?"}
XL_UP = -4162
XL_TO_LEFT = -4159
XL_PASTE_FORMATS = -4122
XL_CALCULATION_MANUAL = -4135
XL_CALCULATION_AUTOMATIC = -4105


def normalize_header(value: Any) -> str:
    return " ".join(str(value or "").strip().split()).casefold()


def normalize_network_id(value: Any) -> str:
    if value is None or isinstance(value, bool):
        return ""
    text = str(value).strip()
    if text.endswith(".0") and text[:-2].isdigit():
        text = text[:-2]
    return text.casefold()


def coerce_excel_date(value: Any) -> date | None:
    if isinstance(value, datetime):
        return value.date()
    if isinstance(value, date):
        return value
    if isinstance(value, (int, float)) and not isinstance(value, bool):
        try:
            return (datetime(1899, 12, 30) + timedelta(days=float(value))).date()
        except (ValueError, OverflowError):
            return None
    text = str(value or "").strip()
    for pattern in ("%m/%d/%Y", "%Y-%m-%d", "%m-%d-%Y"):
        try:
            return datetime.strptime(text, pattern).date()
        except ValueError:
            continue
    return None


def rows_from_range(values: Any) -> list[tuple[Any, ...]]:
    if values in (None, ""):
        return []
    if not isinstance(values, tuple):
        return [(values,)]
    if values and not isinstance(values[0], tuple):
        return [tuple(values)]
    return [tuple(row) for row in values]


def find_header_row(worksheet: Any, required_header: str, maximum_rows: int = 20) -> int:
    last_column = worksheet.UsedRange.Column + worksheet.UsedRange.Columns.Count - 1
    for row_number in range(1, min(maximum_rows, worksheet.UsedRange.Row + worksheet.UsedRange.Rows.Count - 1) + 1):
        values = rows_from_range(worksheet.Range(worksheet.Cells(row_number, 1), worksheet.Cells(row_number, last_column)).Value2)[0]
        if normalize_header(required_header) in {normalize_header(value) for value in values}:
            return row_number
    raise ValueError(f"Could not find header {required_header!r} in the first {maximum_rows} rows of {worksheet.Name!r}.")


def read_headers(worksheet: Any, header_row: int) -> dict[str, int]:
    last_column = worksheet.Cells(header_row, worksheet.Columns.Count).End(XL_TO_LEFT).Column
    values = rows_from_range(worksheet.Range(worksheet.Cells(header_row, 1), worksheet.Cells(header_row, last_column)).Value2)[0]
    headers: dict[str, int] = {}
    duplicates: list[str] = []
    for column, value in enumerate(values, start=1):
        header = normalize_header(value)
        if not header:
            continue
        if header in headers:
            duplicates.append(str(value))
        headers[header] = column
    if duplicates:
        raise ValueError(f"Duplicate nonblank headers in {worksheet.Name!r}: {duplicates}")
    return headers


def resolve_master_sheet(workbook: Any) -> Any:
    matches = [workbook.Worksheets(index) for index in range(1, workbook.Worksheets.Count + 1) if workbook.Worksheets(index).Name.startswith(MASTER_SHEET_PREFIX)]
    if len(matches) != 1:
        raise ValueError(f"Expected exactly one worksheet beginning {MASTER_SHEET_PREFIX!r}; found {[sheet.Name for sheet in matches]}.")
    return matches[0]


def last_data_row(worksheet: Any, key_column: int, header_row: int) -> int:
    row = worksheet.Cells(worksheet.Rows.Count, key_column).End(XL_UP).Row
    return max(header_row, row)


def read_data_rows(worksheet: Any, header_row: int, last_row: int, column_count: int, key_column: int, origin: str) -> list[dict[str, Any]]:
    if last_row <= header_row:
        return []
    raw_rows = rows_from_range(worksheet.Range(worksheet.Cells(header_row + 1, 1), worksheet.Cells(last_row, column_count)).Value2)
    rows = []
    for row_number, values in enumerate(raw_rows, start=header_row + 1):
        if all(value in (None, "") for value in values):
            continue
        network_id = normalize_network_id(values[key_column - 1])
        effective_date = coerce_excel_date(values[0])
        if network_id and effective_date is None:
            raise ValueError(f"{origin} row {row_number} has Network ID {values[key_column - 1]!r} but no valid Effective Date in column A.")
        rows.append({"origin": origin, "row_number": row_number, "values": values, "network_id": network_id, "effective_date": effective_date})
    return rows


def retain_latest_rows(candidates: list[dict[str, Any]]) -> tuple[list[dict[str, Any]], list[dict[str, Any]]]:
    by_network_id: dict[str, list[dict[str, Any]]] = defaultdict(list)
    blank_id_rows = []
    for row in candidates:
        (by_network_id[row["network_id"]] if row["network_id"] else blank_id_rows).append(row)

    retained = list(blank_id_rows)
    removed = []
    ties = []
    for network_id, rows in by_network_id.items():
        latest_date = max(row["effective_date"] for row in rows)
        latest_rows = [row for row in rows if row["effective_date"] == latest_date]
        if len(latest_rows) != 1:
            ties.append(f"{network_id}: " + ", ".join(f"{row['origin']} row {row['row_number']}" for row in latest_rows))
            continue
        retained.append(latest_rows[0])
        removed.extend(row for row in rows if row is not latest_rows[0])
    if ties:
        raise ValueError("Cannot choose a latest record for Network ID(s) with equal maximum Effective Date: " + "; ".join(ties[:20]))
    retained.sort(key=lambda row: (row["origin"] != "master", row["row_number"]))
    return retained, removed


def next_output_path() -> Path:
    OUTPUT_DIR.mkdir(exist_ok=True)
    base = OUTPUT_DIR / f"{MASTER_FILE.stem}_HR_REFRESHED{MASTER_FILE.suffix}"
    candidate = base
    number = 1
    while candidate.exists():
        candidate = base.with_name(f"{base.stem}_{number}{base.suffix}")
        number += 1
    return candidate


def preflight() -> dict[str, Any]:
    if not MASTER_FILE.is_file() or not HR_FILE.is_file():
        raise FileNotFoundError(f"Missing master or HR source: {MASTER_FILE}, {HR_FILE}")
    excel = master_book = source_book = None
    pythoncom.CoInitialize()
    try:
        excel = win32.DispatchEx("Excel.Application")
        excel.Visible = False
        excel.DisplayAlerts = False
        master_book = excel.Workbooks.Open(str(MASTER_FILE.resolve()), ReadOnly=True)
        source_book = excel.Workbooks.Open(str(HR_FILE.resolve()), ReadOnly=True)
        master_sheet = resolve_master_sheet(master_book)
        source_sheet = source_book.Worksheets(SOURCE_SHEET_NAME)
        master_header_row = find_header_row(master_sheet, NETWORK_ID_HEADER)
        source_header_row = find_header_row(source_sheet, NETWORK_ID_HEADER)
        master_headers = read_headers(master_sheet, master_header_row)
        source_headers = read_headers(source_sheet, source_header_row)
        key = normalize_header(NETWORK_ID_HEADER)
        if key not in master_headers or key not in source_headers:
            raise ValueError("Network ID must occur exactly once in both source and master.")
        source_only = sorted(header for header in source_headers if header not in master_headers)
        if source_only:
            raise ValueError(f"Source columns missing from {master_sheet.Name!r}: {source_only}")
        duplicate_column = next((column for header, column in master_headers.items() if header in DUPLICATE_HEADERS), None)
        result = {"master_sheet_name": master_sheet.Name, "source_header_row": source_header_row, "master_header_row": master_header_row, "source_headers": source_headers, "master_headers": master_headers, "source_key_column": source_headers[key], "master_key_column": master_headers[key], "duplicate_column": duplicate_column}
        print(f"Preflight passed: source={source_sheet.Name!r} row {source_header_row}; master={master_sheet.Name!r} row {master_header_row}.")
        print(f"Mapped HR columns: {len(source_headers)}. Optional duplicate control: {'present' if duplicate_column else 'not present'}.")
        return result
    finally:
        if source_book is not None:
            source_book.Close(SaveChanges=False)
        if master_book is not None:
            master_book.Close(SaveChanges=False)
        if excel is not None:
            excel.Quit()
        pythoncom.CoUninitialize()


def analyse_refresh(configuration: dict[str, Any]) -> tuple[list[dict[str, Any]], list[dict[str, Any]]]:
    excel = master_book = source_book = None
    pythoncom.CoInitialize()
    try:
        excel = win32.DispatchEx("Excel.Application")
        excel.Visible = False
        excel.DisplayAlerts = False
        master_book = excel.Workbooks.Open(str(MASTER_FILE.resolve()), ReadOnly=True)
        source_book = excel.Workbooks.Open(str(HR_FILE.resolve()), ReadOnly=True)
        master_sheet = resolve_master_sheet(master_book)
        source_sheet = source_book.Worksheets(SOURCE_SHEET_NAME)
        master_rows = read_data_rows(master_sheet, configuration["master_header_row"], last_data_row(master_sheet, configuration["master_key_column"], configuration["master_header_row"]), max(configuration["master_headers"].values()), configuration["master_key_column"], "master")
        source_rows = read_data_rows(source_sheet, configuration["source_header_row"], last_data_row(source_sheet, configuration["source_key_column"], configuration["source_header_row"]), max(configuration["source_headers"].values()), configuration["source_key_column"], "source")
        retained, removed = retain_latest_rows(master_rows + source_rows)
        print(f"Existing rows: {len(master_rows):,}; source rows: {len(source_rows):,}; retained: {len(retained):,}; superseded: {len(removed):,}.")
        print(f"Blank Network ID rows retained: {sum(not row['network_id'] for row in retained):,}.")
        return retained, removed
    finally:
        if source_book is not None:
            source_book.Close(SaveChanges=False)
        if master_book is not None:
            master_book.Close(SaveChanges=False)
        if excel is not None:
            excel.Quit()
        pythoncom.CoUninitialize()


def refresh_workbook(configuration: dict[str, Any], retained_rows: list[dict[str, Any]]) -> Path:
    output_path = next_output_path()
    shutil.copy2(MASTER_FILE, output_path)
    excel = master_book = None
    original_autofill = None
    succeeded = False
    pythoncom.CoInitialize()
    try:
        excel = win32.DispatchEx("Excel.Application")
        excel.Visible = False
        excel.DisplayAlerts = False
        excel.ScreenUpdating = False
        excel.EnableEvents = False
        excel.Calculation = XL_CALCULATION_MANUAL
        original_autofill = excel.AutoCorrect.AutoFillFormulasInLists
        excel.AutoCorrect.AutoFillFormulasInLists = False
        master_book = excel.Workbooks.Open(str(output_path.resolve()), ReadOnly=False)
        worksheet = resolve_master_sheet(master_book)
        headers = configuration["master_headers"]
        header_row = configuration["master_header_row"]
        key_column = configuration["master_key_column"]
        old_last_row = last_data_row(worksheet, key_column, header_row)
        final_row = header_row + len(retained_rows)
        final_column = max(headers.values())
        if old_last_row > header_row:
            worksheet.Range(worksheet.Cells(old_last_row, 1), worksheet.Cells(old_last_row, final_column)).Copy()
            worksheet.Range(worksheet.Cells(header_row + 1, 1), worksheet.Cells(final_row, final_column)).PasteSpecial(Paste=XL_PASTE_FORMATS)
            worksheet.Application.CutCopyMode = False
            worksheet.Range(worksheet.Cells(header_row + 1, 1), worksheet.Cells(max(old_last_row, final_row), final_column)).ClearContents()
        source_headers = configuration["source_headers"]
        for source_header, source_column in source_headers.items():
            master_column = headers[source_header]
            values = []
            for row in retained_rows:
                if row["origin"] == "source":
                    values.append(row["values"][source_column - 1])
                else:
                    values.append(row["values"][master_column - 1])
            worksheet.Range(worksheet.Cells(header_row + 1, master_column), worksheet.Cells(final_row, master_column)).Value2 = tuple((value,) for value in values)
        duplicate_column = configuration["duplicate_column"]
        if duplicate_column:
            template = worksheet.Cells(header_row + 1, duplicate_column).FormulaR1C1
            if isinstance(template, str) and template.startswith("="):
                worksheet.Range(worksheet.Cells(header_row + 1, duplicate_column), worksheet.Cells(final_row, duplicate_column)).FormulaR1C1 = template
                print("Optional DUPLICATE control formula refreshed.")
            else:
                print("Optional DUPLICATE column has no formula template; preserved without recalculation.")
        for index in range(1, worksheet.ListObjects.Count + 1):
            table = worksheet.ListObjects(index)
            if table.Range.Row == header_row and table.Range.Column <= key_column <= table.Range.Column + table.Range.Columns.Count - 1:
                table.Resize(worksheet.Range(worksheet.Cells(header_row, table.Range.Column), worksheet.Cells(final_row, table.Range.Column + table.Range.Columns.Count - 1)))
                break
        excel.Calculation = XL_CALCULATION_AUTOMATIC
        excel.CalculateFull()
        master_book.Save()
        succeeded = True
        return output_path
    finally:
        if master_book is not None:
            master_book.Close(SaveChanges=succeeded)
        if excel is not None:
            if original_autofill is not None:
                excel.AutoCorrect.AutoFillFormulasInLists = original_autofill
            excel.Quit()
        if not succeeded and output_path.exists():
            output_path.unlink()
        gc.collect()
        pythoncom.CoUninitialize()


def validate_output(output_path: Path, expected_rows: list[dict[str, Any]], configuration: dict[str, Any]) -> None:
    excel = book = None
    pythoncom.CoInitialize()
    try:
        excel = win32.DispatchEx("Excel.Application")
        excel.Visible = False
        excel.DisplayAlerts = False
        book = excel.Workbooks.Open(str(output_path.resolve()), ReadOnly=True)
        worksheet = resolve_master_sheet(book)
        actual = read_data_rows(worksheet, configuration["master_header_row"], last_data_row(worksheet, configuration["master_key_column"], configuration["master_header_row"]), max(configuration["master_headers"].values()), configuration["master_key_column"], "output")
        expected_keys = {(row["network_id"], row["effective_date"]) for row in expected_rows if row["network_id"]}
        actual_keys = {(row["network_id"], row["effective_date"]) for row in actual if row["network_id"]}
        actual_ids = [row["network_id"] for row in actual if row["network_id"]]
        if len(actual_ids) != len(set(actual_ids)) or expected_keys != actual_keys:
            raise AssertionError("Saved workbook does not match the approved unique Network ID + latest Effective Date population.")
        if sum(not row["network_id"] for row in actual) != sum(not row["network_id"] for row in expected_rows):
            raise AssertionError("Saved workbook did not preserve the expected blank Network ID rows.")
        print(f"Post-save validation passed: {len(actual):,} rows; {len(actual_ids):,} unique nonblank Network IDs.")
    finally:
        if book is not None:
            book.Close(SaveChanges=False)
        if excel is not None:
            excel.Quit()
        pythoncom.CoUninitialize()


configuration = preflight()
retained_rows, removed_rows = analyse_refresh(configuration)
print("Review the displayed counts before running the next cell. No workbook has been changed yet.")

Preflight passed: source='To be Copy Pasted from HR' row 4; master='HR HC Combined Jun' row 1.
Mapped HR columns: 24. Optional duplicate control: not present.
Existing rows: 1,624; source rows: 1,400; retained: 1,622; superseded: 1,402.
Blank Network ID rows retained: 0.
Review the displayed counts before running the next cell. No workbook has been changed yet.


In [ ]:
def set_excel_calculation(excel: Any, calculation_mode: int) -> None:
    try:
        excel.Calculation = calculation_mode
    except Exception as error:
        print(f"Excel calculation mode was not changed: {error}")


def refresh_workbook(configuration: dict[str, Any], retained_rows: list[dict[str, Any]]) -> Path:
    output_path = next_output_path()
    shutil.copy2(MASTER_FILE, output_path)
    excel = master_book = None
    original_autofill = None
    succeeded = False
    pythoncom.CoInitialize()
    try:
        excel = win32.DispatchEx("Excel.Application")
        excel.Visible = False
        excel.DisplayAlerts = False
        excel.ScreenUpdating = False
        excel.EnableEvents = False
        set_excel_calculation(excel, XL_CALCULATION_MANUAL)
        try:
            original_autofill = excel.AutoCorrect.AutoFillFormulasInLists
            excel.AutoCorrect.AutoFillFormulasInLists = False
        except Exception as error:
            print(f"Excel table autofill was not changed: {error}")
        master_book = excel.Workbooks.Open(str(output_path.resolve()), ReadOnly=False)
        worksheet = resolve_master_sheet(master_book)
        headers = configuration["master_headers"]
        header_row = configuration["master_header_row"]
        key_column = configuration["master_key_column"]
        old_last_row = last_data_row(worksheet, key_column, header_row)
        final_row = header_row + len(retained_rows)
        final_column = max(headers.values())
        duplicate_column = next((column for header, column in headers.items() if header.rstrip("?") in {"duplicate", "duplicates"}), None)
        duplicate_formula = worksheet.Cells(header_row + 1, duplicate_column).FormulaR1C1 if duplicate_column else None
        if old_last_row > header_row:
            worksheet.Range(worksheet.Cells(old_last_row, 1), worksheet.Cells(old_last_row, final_column)).Copy()
            worksheet.Range(worksheet.Cells(header_row + 1, 1), worksheet.Cells(final_row, final_column)).PasteSpecial(Paste=XL_PASTE_FORMATS)
            worksheet.Application.CutCopyMode = False
            worksheet.Range(worksheet.Cells(header_row + 1, 1), worksheet.Cells(max(old_last_row, final_row), final_column)).ClearContents()
        source_headers = configuration["source_headers"]
        for source_header, source_column in source_headers.items():
            master_column = headers[source_header]
            values = [row["values"][source_column - 1] if row["origin"] == "source" else row["values"][master_column - 1] for row in retained_rows]
            worksheet.Range(worksheet.Cells(header_row + 1, master_column), worksheet.Cells(final_row, master_column)).Value2 = tuple((value,) for value in values)
        if duplicate_column and isinstance(duplicate_formula, str) and duplicate_formula.startswith("="):
            worksheet.Range(worksheet.Cells(header_row + 1, duplicate_column), worksheet.Cells(final_row, duplicate_column)).FormulaR1C1 = duplicate_formula
            print("Optional DUPLICATE control formula refreshed.")
        elif duplicate_column:
            print("Optional DUPLICATE column has no formula template; it remains blank for refreshed rows.")
        for index in range(1, worksheet.ListObjects.Count + 1):
            table = worksheet.ListObjects(index)
            if table.Range.Row == header_row and table.Range.Column <= key_column <= table.Range.Column + table.Range.Columns.Count - 1:
                table.Resize(worksheet.Range(worksheet.Cells(header_row, table.Range.Column), worksheet.Cells(final_row, table.Range.Column + table.Range.Columns.Count - 1)))
                break
        set_excel_calculation(excel, XL_CALCULATION_AUTOMATIC)
        excel.CalculateFull()
        master_book.Save()
        succeeded = True
        return output_path
    finally:
        if master_book is not None:
            master_book.Close(SaveChanges=succeeded)
        if excel is not None:
            if original_autofill is not None:
                excel.AutoCorrect.AutoFillFormulasInLists = original_autofill
            excel.Quit()
        if not succeeded and output_path.exists():
            output_path.unlink()
        gc.collect()
        pythoncom.CoUninitialize()

In [ ]:
def validate_output(output_path: Path, expected_rows: list[dict[str, Any]], configuration: dict[str, Any]) -> None:
    excel = book = None
    pythoncom.CoInitialize()
    gc.collect()
    try:
        excel = win32.DispatchEx("Excel.Application")
        excel.Visible = False
        excel.DisplayAlerts = False
        book = excel.Workbooks.Open(str(output_path.resolve()), ReadOnly=True)
        worksheet = resolve_master_sheet(book)
        actual = read_data_rows(
            worksheet,
            configuration["master_header_row"],
            last_data_row(worksheet, configuration["master_key_column"], configuration["master_header_row"]),
            max(configuration["master_headers"].values()),
            configuration["master_key_column"],
            "output",
        )
        expected_keys = {(row["network_id"], row["effective_date"]) for row in expected_rows if row["network_id"]}
        actual_keys = {(row["network_id"], row["effective_date"]) for row in actual if row["network_id"]}
        actual_ids = [row["network_id"] for row in actual if row["network_id"]]
        if len(actual_ids) != len(set(actual_ids)) or expected_keys != actual_keys:
            raise AssertionError("Saved workbook does not match the approved unique Network ID + latest Effective Date population.")
        if sum(not row["network_id"] for row in actual) != sum(not row["network_id"] for row in expected_rows):
            raise AssertionError("Saved workbook did not preserve the expected blank Network ID rows.")
        print(f"Post-save validation passed: {len(actual):,} rows; {len(actual_ids):,} unique nonblank Network IDs.")
    finally:
        if book is not None:
            try:
                book.Close(SaveChanges=False)
            except Exception:
                pass
        if excel is not None:
            try:
                excel.Quit()
            except Exception:
                pass
        gc.collect()
        pythoncom.CoUninitialize()


configuration["duplicate_column"] = next(
    (column for header, column in configuration["master_headers"].items() if header.rstrip("?") in {"duplicate", "duplicates"}),
    None,
)
print(f"Optional duplicate control: {'present' if configuration['duplicate_column'] else 'not present'}.")

In [ ]:
# Execute only after reviewing the preflight and analysis counts above.
output_path = refresh_workbook(configuration, retained_rows)
validate_output(output_path, retained_rows, configuration)
print(f"Saved validated HR refresh: {output_path}")